# FINAL ANALYTICS

In [0]:
# ============================================================
# 11_FINAL_ANALYTICS
#
# Purpose:
# Create final business analytics datasets
#
# Source:
# ecommerce.gold
#
# Output:
# ecommerce.gold.analytics_*
#
# Used by:
# Power BI / Tableau / Dashboards
#
# ============================================================


from pyspark.sql.functions import *
from pyspark.sql.window import Window



# ============================================================
# 1. Configuration
# ============================================================


gold_schema = "gold"





# ============================================================
# 2. Read Gold Tables
# ============================================================


monthly_sales = spark.table(
    "ecommerce.gold.monthly_sales"
)


customer_summary = spark.table(
    "ecommerce.gold.customer_summary"
)


product_performance = spark.table(
    "ecommerce.gold.product_performance"
)


return_analysis = spark.table(
    "ecommerce.gold.return_analysis"
)


payment_analysis = spark.table(
    "ecommerce.gold.payment_analysis"
)





# ============================================================
# ANALYTIC TABLE 1
#
# Executive KPI Dashboard
#
# Business Questions:
#
# - Total Revenue?
# - Number of Orders?
# - Average Order Value?
#
# ============================================================


orders = spark.table(
    "ecommerce.silver.orders"
)


customers = spark.table(
    "ecommerce.silver.customers"
)


products = spark.table(
    "ecommerce.silver.products"
)



executive_kpi = (

    orders

    .agg(

        sum("total_amount")
        .alias("total_revenue"),


        countDistinct("order_id")
        .alias("total_orders"),


        countDistinct("customer_id")
        .alias("total_customers"),


        avg("total_amount")
        .alias("average_order_value")

    )

)



executive_kpi.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.analytics_executive_kpi"
)



print(
    "Executive KPI Created"
)





# ============================================================
# ANALYTIC TABLE 2
#
# Revenue Trend Analysis
#
# Shows:
# Month over month revenue
#
# ============================================================



revenue_trend_window = Window.orderBy(
    "month"
)



revenue_trend = (

    monthly_sales


    .withColumn(

        "previous_month",

        lag(
            "total_revenue"
        )
        .over(revenue_trend_window)

    )


    .withColumn(

        "growth_percentage",

        (

            (

            col("total_revenue")
            -
            col("previous_month")

            )

            /

            col("previous_month")

        )

        *

        100

    )

)



revenue_trend.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.analytics_revenue_trend"
)



print(
    "Revenue Trend Created"
)





# ============================================================
# ANALYTIC TABLE 3
#
# Customer Segmentation
#
# Segment customers:
#
# HIGH VALUE
# MEDIUM VALUE
# LOW VALUE
#
# ============================================================



customer_segment = (

    customer_summary


    .withColumn(

        "customer_segment",

        when(

            col("total_spent") >= 100000,

            "HIGH VALUE"

        )

        .when(

            col("total_spent") >= 50000,

            "MEDIUM VALUE"

        )

        .otherwise(

            "LOW VALUE"

        )

    )

)



customer_segment.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.analytics_customer_segment"
)



print(
    "Customer Segmentation Created"
)





# ============================================================
# ANALYTIC TABLE 4
#
# Top Customers
#
# ============================================================



top_customers = (

    customer_summary


    .orderBy(

        col("total_spent")
        .desc()

    )


    .limit(100)

)



top_customers.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.analytics_top_customers"
)



print(
    "Top Customers Created"
)





# ============================================================
# ANALYTIC TABLE 5
#
# Top Selling Products
#
# ============================================================



top_products = (

    product_performance


    .orderBy(

        col("revenue")
        .desc()

    )


    .limit(100)

)



top_products.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.analytics_top_products"
)



print(
    "Top Products Created"
)





# ============================================================
# ANALYTIC TABLE 6
#
# Category Performance
#
# ============================================================



category_analysis = (

    product_performance


    .groupBy(
        "category"
    )


    .agg(

        sum("revenue")
        .alias("category_revenue"),


        sum("units_sold")
        .alias("units_sold"),


        countDistinct("product_id")
        .alias("number_of_products")

    )


)



category_analysis.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.analytics_category"
)



print(
    "Category Analysis Created"
)





# ============================================================
# ANALYTIC TABLE 7
#
# Return Rate Analysis
#
# ============================================================



return_summary = (

    return_analysis


    .groupBy(
        "category"
    )


    .agg(

        sum("total_returns")
        .alias("total_returns"),


        sum("total_refunded")
        .alias("total_refunded")

    )

)



return_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.analytics_returns"
)



print(
    "Return Analytics Created"
)





# ============================================================
# ANALYTIC TABLE 8
#
# Payment Insights
#
# ============================================================



payment_analysis.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.analytics_payment"
)



print(
    "Payment Analytics Created"
)





# ============================================================
# FINAL VALIDATION
# ============================================================


print(
    "FINAL ANALYTICS TABLES"
)



spark.sql(
"""
SHOW TABLES IN ecommerce.gold
"""
).show(
    truncate=False
)

Executive KPI Created


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Revenue Trend Created
Customer Segmentation Created
Top Customers Created
Top Products Created
Category Analysis Created
Return Analytics Created
Payment Analytics Created
FINAL ANALYTICS TABLES
+--------+--------------------------+-----------+
|database|tableName                 |isTemporary|
+--------+--------------------------+-----------+
|gold    |analytics_category        |false      |
|gold    |analytics_customer_segment|false      |
|gold    |analytics_executive_kpi   |false      |
|gold    |analytics_payment         |false      |
|gold    |analytics_returns         |false      |
|gold    |analytics_revenue_trend   |false      |
|gold    |analytics_top_customers   |false      |
|gold    |analytics_top_products    |false      |
|gold    |customer_summary          |false      |
|gold    |monthly_sales             |false      |
|gold    |payment_analysis          |false      |
|gold    |product_performance       |false      |
|gold    |return_analysis           |false      |
|gold

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
%sql
SELECT
    month,
    total_revenue
FROM ecommerce.gold.analytics_revenue_trend
ORDER BY month;

month,total_revenue
2024-01,10505341.82
2024-02,9844555.08
2024-03,10319128.58
2024-04,9932687.73
2024-05,10785014.29
2024-06,10068460.40
2024-07,10297669.12
2024-08,10551482.10
2024-09,10280113.75
2024-10,10755209.33


In [0]:
%sql
SELECT
first_name,
last_name,
total_spent
FROM ecommerce.gold.analytics_top_customers
LIMIT 10;

first_name,last_name,total_spent
James,Gamble,459110.63
Jennifer,Soto,436617.42
Bryan,Parker,421803.97
Edward,Jones,414413.85
Teresa,Taylor,413103.37
Kimberly,Perry,407165.53
Joshua,Smith,401726.55
Samuel,Mcdonald,398792.32
William,Daniels,394983.17
Jonathan,Levy,394708.21


In [0]:
%sql
SELECT
product_name,
revenue
FROM ecommerce.gold.analytics_top_products
LIMIT 10;

product_name,revenue
L'Oreal Job 527,139046.56
Philips Eat 600,129038.21
L'Oreal Laugh 834,128296.22
Wilson Building 795,124365.81
Panasonic Blue 680,122922.72
Panasonic Lawyer 971,122559.98
Apple Might 784,122023.51
HarperCollins Machine 988,121496.89
Penguin Perhaps 598,118804.37
Wilson Now 421,117827.04


In [0]:
%sql
SELECT
  city,
  product_id AS top_product_id,
  product_name AS top_product_name,
  product_revenue AS top_product_revenue,
  product_orders AS total_orders
FROM (
  SELECT
    city,
    ecommerce.bronze.orders.product_id,
    products.product_name,
    COUNT(DISTINCT order_id) AS product_orders,
    SUM(total_amount) AS product_revenue,
    ROW_NUMBER() OVER (PARTITION BY city ORDER BY SUM(total_amount) DESC) AS rn
  FROM ecommerce.bronze.orders
  LEFT JOIN ecommerce.silver.products
    ON ecommerce.bronze.orders.product_id = products.product_id
  GROUP BY city, ecommerce.bronze.orders.product_id, products.product_name
)
WHERE rn = 1
ORDER BY total_orders DESC

city,top_product_id,top_product_name,top_product_revenue,total_orders
Rawalpindi,4093,IKEA Toward 872,38311.34,13
Lahore,4274,Under Armour Employee 139,39136.619999999995,8
Islamabad,1257,Zara Talk 266,36968.24,8
Peshawar,15,Tefal Begin 103,44527.94,7
Faisalabad,3962,Adidas Source 870,38243.1,7
Quetta,2251,HarperCollins Machine 988,36317.33,7
Karachi,1560,IKEA Season 350,37354.14,6
Multan,4470,Wilson Building 795,45231.17999999999,6


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select * from ecommerce.bronze.products;

brand,category,cost_price,launch_date,operation,operation_timestamp,price,product_id,product_name,rating,status,stock_quantity,supplier,_rescued_data
Penguin,Books,1113.62,2020-04-12,INSERT,2026-06-21T09:37:32Z,1361.34,1,Penguin Purpose 242,3.2,Active,1508,Penguin Supplier,null
Wilson,Sports,52.52,2024-09-24,INSERT,2026-04-23T05:53:39Z,60.23,2,Wilson Brother 338,4.2,Active,1034,Wilson Supplier,null
HarperCollins,Books,632.18,2026-02-10,INSERT,2024-07-14T23:01:57Z,837.39,3,HarperCollins Ago 384,4.7,Active,1657,HarperCollins Supplier,null
Puma,Fashion,513.67,2024-09-27,INSERT,2024-01-28T04:17:10Z,604.97,4,Puma Site 881,3.2,Active,689,Puma Supplier,null
Tefal,Home,1211.66,2022-12-19,INSERT,2024-11-17T13:22:13Z,1774.92,5,Tefal Face 649,4.9,Active,255,Tefal Supplier,null
Yonex,Sports,868.14,2024-01-21,INSERT,2025-10-08T06:21:28Z,1260.79,6,Yonex Election 146,3.5,Active,1354,Yonex Supplier,null
Samsung,Electronics,573.29,2021-02-17,INSERT,2024-05-08T02:24:01Z,760.59,7,Samsung Dog 954,3.3,Active,747,Samsung Supplier,null
O'Reilly,Books,915.65,2020-10-19,INSERT,2025-04-16T05:01:07Z,1085.57,8,O'Reilly Chair 846,3.3,Active,501,O'Reilly Supplier,null
HarperCollins,Books,333.32,2026-03-31,INSERT,2025-07-05T13:21:12Z,420.7,9,HarperCollins Since 886,3.1,Active,1589,HarperCollins Supplier,null
Panasonic,Home,103.95,2023-01-01,INSERT,2024-02-13T20:30:48Z,161.81,10,Panasonic Blue 680,4.4,Active,1794,Panasonic Supplier,null
